# 时序差分算法学习笔记

本文基于《动手学强化学习》第 5 章“时序差分算法”整理，原文链接：<https://hrl.boyuai.com/chapter/1/%E6%97%B6%E5%BA%8F%E5%B7%AE%E5%88%86%E7%AE%97%E6%B3%95/>。

这份 notebook 的目标不是把原文再抄一遍，而是把初学者最容易卡住的地方讲透：

- 为什么学完动态规划之后，还要继续学时序差分（TD）？
- `TD target`、`TD error`、`bootstrapping` 到底分别是什么意思？
- Sarsa、多步 Sarsa、Q-learning 的更新目标究竟差在哪里？
- 为什么在悬崖漫步里，Sarsa 更保守，Q-learning 却更喜欢贴着悬崖走？

如果你是第一次接触这一章，建议这样读：

1. 先看第 1 到第 3 节，先把直觉建立起来。
2. 再看悬崖漫步环境，搞清楚状态编号和奖励规则。
3. 最后对照三种算法的更新公式和代码实现。

## 1. 为什么要从 DP 走到 TD？

第 4 章的动态规划（DP）有一个很强的前提：**环境模型已知**。也就是你不仅知道有哪些状态和动作，还知道在状态 `s` 做动作 `a` 之后，会以什么概率去哪个下一状态、拿到多少奖励。

这就像你手里已经有一本“环境说明书”，所以可以直接列方程、直接算价值函数。

但真实任务往往没有这么理想。比如游戏、机器人控制、复杂推荐系统，你不可能先把完整的状态转移概率表写出来。于是就只能换一种思路：**边和环境交互，边从样本里学习。** 这类方法就叫无模型强化学习（model-free reinforcement learning）。

这一章里的时序差分方法（Temporal Difference, TD），可以理解成夹在 Monte Carlo 和 DP 之间的一座桥：

- **像 Monte Carlo**：它不需要提前知道环境模型，可以直接用采样到的数据学习。
- **像 DP**：它不会傻等整条序列结束，而是会利用“下一状态的当前估计值”来更新当前状态的估计。

如果用一句更口语化的话概括：

> DP 像拿着完整地图算最优路线；Monte Carlo 像整趟路走完了再复盘；TD 像走一步就根据眼前情况和已有经验修正一次导航。

## 2. TD(0) 到底在做什么？

先回忆蒙特卡洛方法的核心想法：在状态 `s_t` 出现后，要等整条序列结束，拿到完整回报 `G_t`，再去更新 `V(s_t)`：

$$
V(s_t) \leftarrow V(s_t) + \alpha \bigl[G_t - V(s_t)\bigr]
$$

它的优点是目标 `G_t` 很“真”，因为它来自真实后续回报；缺点是必须等到回合结束，更新太晚，而且方差通常比较大。

TD(0) 把目标换成了一步目标：

$$
V(s_t) \leftarrow V(s_t) + \alpha \Bigl[r_t + \gamma V(s_{t+1}) - V(s_t)\Bigr]
$$

这个式子建议拆成 3 块看：

- `V(s_t)`：你当前对这个状态的旧看法。
- `r_t + \gamma V(s_{t+1})`：你根据“当前奖励 + 下一状态估计值”构造出来的新证据，也叫 **TD target**。
- `r_t + \gamma V(s_{t+1}) - V(s_t)`：新证据和旧看法之间的差，也叫 **TD error**。

所以 TD 更新本质上就是：

> 如果新证据比旧估计高，就把当前价值往上拉一点；如果更低，就往下修一点。

举个很小的数字例子：假设此时 `V(s_t)=3`，当前拿到奖励 `r_t=-1`，下一状态估计值 `V(s_{t+1})=5`，折扣因子 `\gamma=0.9`。那么：

$$
\text{TD target} = -1 + 0.9 \times 5 = 3.5
$$

这说明“新的证据”认为当前状态大约值 `3.5`，而你原先只估计成 `3`，所以 TD 误差就是 `0.5`，接下来会往上更新。

这里最容易让人困惑的词叫 **bootstrapping（自举）**。它的意思不是“凭空猜”，而是：

> 我不是等到未来全部发生后再算，而是先用自己当前对下一状态的估计，来帮助更新当前状态。

因此 TD 的特点也就很鲜明了：

- 更新快，因为每走一步就能学；
- 方差更小，因为不需要把整条轨迹的随机性都累加起来；
- 但会有偏差，因为 `V(s_{t+1})` 本身也只是一个估计值。

## 3. 为什么控制问题要学 `Q(s,a)`？

如果只是评估一个固定策略好不好，学 `V(s)` 就够了，因为它回答的是“站在这个状态上整体值多少钱”。

但如果你真正关心的是“下一步到底该选哪个动作”，那就必须学动作价值函数 `Q(s,a)`，因为它回答的是：

> 在状态 `s` 先做动作 `a`，然后再继续行动，这条路值多少钱？

一旦有了 `Q(s,a)`，策略提升就自然了：哪个动作的 `Q` 值大，就更倾向于选哪个动作。

不过这里立刻会撞上一个经典问题：**如果永远只选当前看起来最好的动作，会不会太早把自己困住？**

答案是会。因为刚开始的 `Q` 值几乎全是瞎猜，如果你完全贪心，很可能只是碰巧某个动作前几次样本好看，就一直选它，导致很多状态动作对根本没机会被探索。

因此原文使用的是 `\epsilon`-贪心策略：

- 以较大概率选择当前 `Q` 值最大的动作，负责“利用”；
- 以较小概率随机选动作，负责“探索”。

这一步非常关键，因为后面 Sarsa 和 Q-learning 的差异，恰恰就出在：**它们更新时到底把“下一步会怎么选动作”看成什么。**

## 4. 悬崖漫步环境：先把地图看懂

原文继续沿用 Cliff Walking（悬崖漫步）作为示例环境。这个环境的关键不是复杂，而是特别适合观察“保守策略”和“激进策略”的区别。

可以把它想成一个 `4 x 12` 的网格，左下角是起点 `S`，右下角是终点 `G`，底边中间一整排都是悬崖：

```text
(0,0) ........................ (0,11)
(1,0) ........................ (1,11)
(2,0) ........................ (2,11)
S      C   C   C   C   C   C   C   C   C   C   G
```

其中：

- `S` 在 `(3, 0)`，也就是左下角；
- `G` 在 `(3, 11)`，也就是右下角；
- `C` 是悬崖，对应 `(3, 1)` 到 `(3, 10)`；
- 普通移动奖励是 `-1`；
- 掉下悬崖奖励是 `-100`，并且回合终止。

代码里不会一直用二维坐标，而是会把 `(row, col)` 压平成一个一维编号：

$$
state = row \times ncol + col
$$

所以在 `4 x 12` 的地图里：

- 起点 `(3, 0)` 对应状态 `36`；
- 终点 `(3, 11)` 对应状态 `47`；
- 悬崖区对应状态 `37` 到 `46`。

下面这段代码保留了原文的核心环境设定，但我把一些终止条件写得更显式，这样后面读更新公式时会更顺。

In [ ]:
import os

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

import matplotlib.pyplot as plt
import numpy as np

np.random.seed(0)


class CliffWalkingEnv:
    # 4x12 悬崖漫步环境：左下角起点，右下角终点。
    def __init__(self, ncol=12, nrow=4):
        self.ncol = ncol
        self.nrow = nrow
        self.reset()

    def step(self, action):
        change = [(0, -1), (0, 1), (-1, 0), (1, 0)]
        dx, dy = change[action]
        self.x = min(self.ncol - 1, max(0, self.x + dx))
        self.y = min(self.nrow - 1, max(0, self.y + dy))

        next_state = self.y * self.ncol + self.x
        reward = -1
        done = False

        if self.y == self.nrow - 1 and self.x > 0:
            done = True
            if self.x != self.ncol - 1:
                reward = -100
        return next_state, reward, done

    def reset(self):
        self.x = 0
        self.y = self.nrow - 1
        return self.y * self.ncol + self.x


def print_agent(agent, env, action_meaning, disaster=None, end=None):
    disaster = set(disaster or [])
    end = set(end or [])
    for row in range(env.nrow):
        for col in range(env.ncol):
            state = row * env.ncol + col
            if state in disaster:
                print('****', end=' ')
            elif state in end:
                print('EEEE', end=' ')
            else:
                best = agent.best_action(state)
                pi_str = ''.join(action_meaning[i] if best[i] else 'o' for i in range(len(action_meaning)))
                print(pi_str, end=' ')
        print()


def plot_returns(returns_dict, window=10):
    plt.figure(figsize=(10, 4))
    for name, returns in returns_dict.items():
        returns = np.asarray(returns)
        if len(returns) >= window:
            kernel = np.ones(window) / window
            smooth = np.convolve(returns, kernel, mode='valid')
            x = np.arange(window - 1, len(returns))
            plt.plot(x, smooth, label=f'{name} ({window} 回合滑动平均)')
        else:
            plt.plot(returns, label=name)
    plt.xlabel('Episodes')
    plt.ylabel('Return')
    plt.title('Cliff Walking')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()


env = CliffWalkingEnv()
action_meaning = ['^', 'v', '<', '>']

## 5. Sarsa：学的是“当前这套会探索的行为”

Sarsa 的名字来自更新时用到的五元组：

$$
(s_t, a_t, r_t, s_{t+1}, a_{t+1})
$$

它的更新公式是：

$$
Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha \Bigl[r_t + \gamma Q(s_{t+1}, a_{t+1}) - Q(s_t, a_t)\Bigr]
$$

这条公式最关键的地方不是形式，而是 `a_{t+1}` 的含义：

> 它不是“理论上最好的下一个动作”，而是“按照当前行为策略，下一步实际会选到的动作”。

这就意味着：如果你当前用的是 `\epsilon`-贪心策略，那么 Sarsa 学到的就是这套带有探索噪声的策略本身的价值。

所以在悬崖漫步里，Sarsa 往往会更保守。原因并不神秘：

- 它知道自己下一步还有一定概率随机乱走；
- 一旦靠悬崖太近，这一点点随机性就可能把自己送进 `-100`；
- 因而即便“理论最短路”更短，它也会倾向于选择更安全的路线。

这就是所谓的 **on-policy（在线策略）**：

> 你拿什么策略去采样，就用这套策略对应的目标去更新。行为策略和学习目标是同一个。

In [ ]:
class Sarsa:
    def __init__(self, ncol, nrow, epsilon, alpha, gamma, n_action=4):
        self.Q_table = np.zeros((nrow * ncol, n_action))
        self.n_action = n_action
        self.epsilon = epsilon
        self.alpha = alpha
        self.gamma = gamma

    def take_action(self, state):
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_action)
        return int(np.argmax(self.Q_table[state]))

    def best_action(self, state):
        q_max = np.max(self.Q_table[state])
        return [int(self.Q_table[state, a] == q_max) for a in range(self.n_action)]

    def update(self, s0, a0, r, s1, a1, done):
        td_target = r if done else r + self.gamma * self.Q_table[s1, a1]
        td_error = td_target - self.Q_table[s0, a0]
        self.Q_table[s0, a0] += self.alpha * td_error


def train_sarsa(env, agent, num_episodes=500):
    returns = []
    for _ in range(num_episodes):
        episode_return = 0
        state = env.reset()
        action = agent.take_action(state)
        done = False

        while not done:
            next_state, reward, done = env.step(action)
            next_action = agent.take_action(next_state) if not done else 0
            agent.update(state, action, reward, next_state, next_action, done)
            state, action = next_state, next_action
            episode_return += reward

        returns.append(episode_return)
    return returns

In [ ]:
epsilon = 0.1
alpha = 0.1
gamma = 0.9
num_episodes = 500

sarsa_agent = Sarsa(env.ncol, env.nrow, epsilon=epsilon, alpha=alpha, gamma=gamma)
sarsa_returns = train_sarsa(env, sarsa_agent, num_episodes=num_episodes)

plot_returns({'Sarsa': sarsa_returns})
print('Sarsa 最终收敛得到的策略为：')
print_agent(sarsa_agent, env, action_meaning, disaster=list(range(37, 47)), end=[47])

## 6. 多步 Sarsa：在 TD 和 Monte Carlo 之间找折中

原文第 5.4 节想回答的问题很自然：

- 单步 TD 更新得很快，但有偏；
- Monte Carlo 用真实完整回报，没偏，但方差大而且要等回合结束；
- 那能不能折中一下？

答案就是 **多步时序差分**。对于多步 Sarsa，它把单步目标

$$
r_t + \gamma Q(s_{t+1}, a_{t+1})
$$

替换成了一个更长的目标：

$$
G_t^{(n)} = r_t + \gamma r_{t+1} + \cdots + \gamma^{n-1} r_{t+n-1} + \gamma^n Q(s_{t+n}, a_{t+n})
$$

然后更新：

$$
Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha \bigl[G_t^{(n)} - Q(s_t, a_t)\bigr]
$$

这条公式可以这样理解：

- 前面 `n` 步，尽量多看一些真实奖励；
- 但又不必等整条回合结束；
- 走到第 `n` 步后，再用一个 bootstrap 项 `Q(s_{t+n}, a_{t+n})` 接上去。

于是它就在“偏差”和“方差”之间做了折中：

- `n=1` 时，它就退化成普通 Sarsa；
- `n` 很大时，它越来越像 Monte Carlo；
- 一般来说，适当增大 `n` 会让学习更稳定，也更快利用后续奖励信息。

这一段的代码实现最容易难住人的地方，是变量 `tau = t - n + 1`。它的直觉解释其实很简单：

> 当前时刻是 `t`，说明我终于攒够了某个历史状态动作对往后 `n` 步的信息，所以该轮到 `tau` 那一步结算了。

下面的实现比原文多做了一点显式处理：回合结束时，我会把最后不足 `n` 步的尾巴也完整结算掉，这样逻辑更完整，也更方便自己复盘。

In [ ]:
class NStepSarsa(Sarsa):
    def __init__(self, n, ncol, nrow, epsilon, alpha, gamma, n_action=4):
        super().__init__(ncol, nrow, epsilon, alpha, gamma, n_action)
        self.n = n


def train_n_step_sarsa(env, agent, num_episodes=500):
    returns = []
    for _ in range(num_episodes):
        episode_return = 0
        states = [env.reset()]
        actions = [agent.take_action(states[0])]
        rewards = [0.0]
        T = np.inf
        t = 0

        while True:
            if t < T:
                next_state, reward, done = env.step(actions[t])
                states.append(next_state)
                rewards.append(reward)
                episode_return += reward

                if done:
                    T = t + 1
                else:
                    actions.append(agent.take_action(next_state))

            tau = t - agent.n + 1
            if tau >= 0:
                upper = tau + agent.n if T == np.inf else min(tau + agent.n, int(T))
                G = 0.0
                for i in range(tau + 1, upper + 1):
                    G += (agent.gamma ** (i - tau - 1)) * rewards[i]
                if tau + agent.n < T:
                    G += (agent.gamma ** agent.n) * agent.Q_table[states[tau + agent.n], actions[tau + agent.n]]

                s_tau = states[tau]
                a_tau = actions[tau]
                agent.Q_table[s_tau, a_tau] += agent.alpha * (G - agent.Q_table[s_tau, a_tau])

            if tau == T - 1:
                break
            t += 1

        returns.append(episode_return)
    return returns

In [ ]:
n_step = 5

nstep_agent = NStepSarsa(n_step, env.ncol, env.nrow, epsilon=epsilon, alpha=alpha, gamma=gamma)
nstep_returns = train_n_step_sarsa(env, nstep_agent, num_episodes=num_episodes)

plot_returns({'5-step Sarsa': nstep_returns})
print('5 步 Sarsa 最终收敛得到的策略为：')
print_agent(nstep_agent, env, action_meaning, disaster=list(range(37, 47)), end=[47])

## 7. Q-learning：行为可以探索，但更新盯着“下一步最优”

Q-learning 和 Sarsa 最大的不同，只藏在一个地方：**下一个状态的目标值到底怎么取。**

Q-learning 的更新公式是：

$$
Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha \Bigl[r_t + \gamma \max_a Q(s_{t+1}, a) - Q(s_t, a_t)\Bigr]
$$

把它和 Sarsa 对比一下：

- **Sarsa** 用的是 `Q(s_{t+1}, a_{t+1})`，也就是“当前行为策略实际会选到的下一个动作”。
- **Q-learning** 用的是 `\max_a Q(s_{t+1}, a)`，也就是“如果下一步完全贪心，理论上最好的动作价值”。

所以 Q-learning 的逻辑是：

> 我跟环境交互时可以继续带探索，但我更新时始终把目标看成“下一步一定会挑最好的动作”。

这就是 **off-policy（离线策略）** 的核心：

- **行为策略**：你实际上拿什么策略去采样数据；
- **目标策略**：你希望学习到哪套策略；
- 两者可以不同。

也正因为如此，Q-learning 往往会在悬崖漫步里学到更激进的路径。它学习的目标更接近“真正的最优贪心策略”，所以更愿意贴着悬崖走最短路。

但这也带来一个特别容易误解的现象：

> 训练过程中，Q-learning 的回报不一定比 Sarsa 高；可它最终学到的目标策略反而可能更优。

原因就在于训练时你通常仍然用 `\epsilon`-贪心去采样，所以它在悬崖边探索时更容易掉下去，导致训练曲线看起来吃亏；然而它真正学的目标是“没有探索噪声时的贪心策略”，因此最后打印出来的策略会更贴最优路线。

In [ ]:
class QLearning:
    def __init__(self, ncol, nrow, epsilon, alpha, gamma, n_action=4):
        self.Q_table = np.zeros((nrow * ncol, n_action))
        self.n_action = n_action
        self.epsilon = epsilon
        self.alpha = alpha
        self.gamma = gamma

    def take_action(self, state):
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_action)
        return int(np.argmax(self.Q_table[state]))

    def best_action(self, state):
        q_max = np.max(self.Q_table[state])
        return [int(self.Q_table[state, a] == q_max) for a in range(self.n_action)]

    def update(self, s0, a0, r, s1, done):
        td_target = r if done else r + self.gamma * np.max(self.Q_table[s1])
        td_error = td_target - self.Q_table[s0, a0]
        self.Q_table[s0, a0] += self.alpha * td_error


def train_q_learning(env, agent, num_episodes=500):
    returns = []
    for _ in range(num_episodes):
        episode_return = 0
        state = env.reset()
        done = False

        while not done:
            action = agent.take_action(state)
            next_state, reward, done = env.step(action)
            agent.update(state, action, reward, next_state, done)
            state = next_state
            episode_return += reward

        returns.append(episode_return)
    return returns

In [ ]:
qlearning_agent = QLearning(env.ncol, env.nrow, epsilon=epsilon, alpha=alpha, gamma=gamma)
qlearning_returns = train_q_learning(env, qlearning_agent, num_episodes=num_episodes)

plot_returns({'Q-learning': qlearning_returns})
print('Q-learning 最终收敛得到的策略为：')
print_agent(qlearning_agent, env, action_meaning, disaster=list(range(37, 47)), end=[47])

In [ ]:
plot_returns({
    'Sarsa': sarsa_returns,
    '5-step Sarsa': nstep_returns,
    'Q-learning': qlearning_returns,
})

## 8. 一页总结：这三种算法到底怎么区分？

| 算法 | 更新目标 | 属于什么 | 直觉特点 |
| --- | --- | --- | --- |
| Sarsa | `r + \gamma Q(s', a')` | on-policy | 学的是当前这套带探索的行为，通常更保守 |
| `n` 步 Sarsa | `n` 步真实奖励 `+ \gamma^n Q(s_{t+n}, a_{t+n})` | on-policy | 在偏差和方差之间折中，常常收敛更快 |
| Q-learning | `r + \gamma \max_a Q(s', a)` | off-policy | 更新时始终盯着“下一步最优”，更激进 |

如果只记一句话，我会记：

> Sarsa 关心“我现在这样做，下一步实际还会怎么走”；Q-learning 关心“下一步最理想应该怎么走”。

再补 4 个常见疑问：

**1. TD 为什么不用等回合结束？**
因为它直接拿一步奖励和下一状态估计值来构造目标，不需要完整回报。

**2. TD 误差是真实误差吗？**
不是。它只是“当前一步目标”和“当前旧估计”的差。它是学习信号，不是真值误差。

**3. 为什么 Q-learning 训练曲线可能更差，但最终策略更好？**
因为训练时的行为策略通常还在探索，会掉下悬崖；但它学习的目标是更贪心、更接近最优的策略。

**4. 这一章和后面的 DQN 是什么关系？**
DQN 本质上就是把这里的表格型 Q-learning，换成了神经网络近似的 Q 函数，再加上一些稳定训练的技巧。主线没有变。

原文第 5.7 节还给出了 Q-learning 收敛性的扩展阅读证明。这部分更偏数学分析，第一次学这一章时，建议先把本 notebook 里的直觉、公式和代码逻辑真正打通，再回头补证明。